In [0]:
-- ============================================================
-- BRONZE LAYER VALIDATION
-- Clinical Trial Intelligence Platform
-- ============================================================
--
-- Purpose:
-- 1. Verify Bronze dataset inventory
-- 2. Validate row counts
-- 3. Validate ingestion lineage
-- 4. Validate technical metadata completeness
-- 5. Inspect repeated business keys in append-heavy feeds
--
-- NOTE:
-- Bronze preserves raw ingestion history.
-- Business-level cleansing/deduplication is performed in Silver.
-- ============================================================


-- ============================================================
-- 1. VERIFY BRONZE DATASET INVENTORY
-- ============================================================

SHOW TABLES IN clinical_trial_intelligence.bronze;



-- ============================================================
-- 2. ROW COUNT VALIDATION
-- STREAMING TABLES
-- ============================================================

SELECT 'edc_subjects' AS table_name, COUNT(*) AS row_count
FROM clinical_trial_intelligence.bronze.edc_subjects

UNION ALL

SELECT 'edc_visits', COUNT(*)
FROM clinical_trial_intelligence.bronze.edc_visits

UNION ALL

SELECT 'lab_results', COUNT(*)
FROM clinical_trial_intelligence.bronze.lab_results

UNION ALL

SELECT 'safety_adverse_events', COUNT(*)
FROM clinical_trial_intelligence.bronze.safety_adverse_events

ORDER BY table_name;



-- ============================================================
-- 3. ROW COUNT VALIDATION
-- MATERIALIZED VIEWS
-- ============================================================

SELECT 'ctms_sites' AS table_name, COUNT(*) AS row_count
FROM clinical_trial_intelligence.bronze.ctms_sites

UNION ALL

SELECT 'ctms_studies', COUNT(*)
FROM clinical_trial_intelligence.bronze.ctms_studies

UNION ALL

SELECT 'master_institutions', COUNT(*)
FROM clinical_trial_intelligence.bronze.master_institutions

UNION ALL

SELECT 'master_products', COUNT(*)
FROM clinical_trial_intelligence.bronze.master_products

UNION ALL

SELECT 'master_sponsors', COUNT(*)
FROM clinical_trial_intelligence.bronze.master_sponsors

UNION ALL

SELECT 'protocol_study_arms', COUNT(*)
FROM clinical_trial_intelligence.bronze.protocol_study_arms

UNION ALL

SELECT 'ref_country_region', COUNT(*)
FROM clinical_trial_intelligence.bronze.ref_country_region

UNION ALL

SELECT 'ref_diagnosis_mapping', COUNT(*)
FROM clinical_trial_intelligence.bronze.ref_diagnosis_mapping

UNION ALL

SELECT 'ref_geography', COUNT(*)
FROM clinical_trial_intelligence.bronze.ref_geography

UNION ALL

SELECT 'ref_lab_test', COUNT(*)
FROM clinical_trial_intelligence.bronze.ref_lab_test

UNION ALL

SELECT 'ref_severity_mapping', COUNT(*)
FROM clinical_trial_intelligence.bronze.ref_severity_mapping

UNION ALL

SELECT 'ref_sex_mapping', COUNT(*)
FROM clinical_trial_intelligence.bronze.ref_sex_mapping

UNION ALL

SELECT 'ref_unit_mapping', COUNT(*)
FROM clinical_trial_intelligence.bronze.ref_unit_mapping

ORDER BY table_name;



-- ============================================================
-- 4. VALIDATE SOURCE FILE LINEAGE
-- STREAMING TABLES
-- ============================================================

SELECT
    'edc_subjects' AS table_name,
    COUNT(DISTINCT _source_file) AS files_seen,
    COUNT(*) AS rows_loaded,
    MIN(_ingestion_ts) AS first_ingestion,
    MAX(_ingestion_ts) AS latest_ingestion
FROM clinical_trial_intelligence.bronze.edc_subjects

UNION ALL

SELECT
    'edc_visits',
    COUNT(DISTINCT _source_file),
    COUNT(*),
    MIN(_ingestion_ts),
    MAX(_ingestion_ts)
FROM clinical_trial_intelligence.bronze.edc_visits

UNION ALL

SELECT
    'lab_results',
    COUNT(DISTINCT _source_file),
    COUNT(*),
    MIN(_ingestion_ts),
    MAX(_ingestion_ts)
FROM clinical_trial_intelligence.bronze.lab_results

UNION ALL

SELECT
    'safety_adverse_events',
    COUNT(DISTINCT _source_file),
    COUNT(*),
    MIN(_ingestion_ts),
    MAX(_ingestion_ts)
FROM clinical_trial_intelligence.bronze.safety_adverse_events

ORDER BY table_name;



-- ============================================================
-- 5. BRONZE METADATA COMPLETENESS
-- STREAMING TABLES
-- ============================================================

SELECT
    'edc_subjects' AS table_name,
    COUNT(*) AS total_rows,
    COUNT_IF(_source_file IS NULL) AS missing_source_file,
    COUNT_IF(_source_file_name IS NULL) AS missing_source_file_name,
    COUNT_IF(_source_file_modification_ts IS NULL)
        AS missing_file_modification_ts,
    COUNT_IF(_ingestion_ts IS NULL) AS missing_ingestion_ts,
    COUNT_IF(_ingestion_date IS NULL) AS missing_ingestion_date
FROM clinical_trial_intelligence.bronze.edc_subjects

UNION ALL

SELECT
    'edc_visits',
    COUNT(*),
    COUNT_IF(_source_file IS NULL),
    COUNT_IF(_source_file_name IS NULL),
    COUNT_IF(_source_file_modification_ts IS NULL),
    COUNT_IF(_ingestion_ts IS NULL),
    COUNT_IF(_ingestion_date IS NULL)
FROM clinical_trial_intelligence.bronze.edc_visits

UNION ALL

SELECT
    'lab_results',
    COUNT(*),
    COUNT_IF(_source_file IS NULL),
    COUNT_IF(_source_file_name IS NULL),
    COUNT_IF(_source_file_modification_ts IS NULL),
    COUNT_IF(_ingestion_ts IS NULL),
    COUNT_IF(_ingestion_date IS NULL)
FROM clinical_trial_intelligence.bronze.lab_results

UNION ALL

SELECT
    'safety_adverse_events',
    COUNT(*),
    COUNT_IF(_source_file IS NULL),
    COUNT_IF(_source_file_name IS NULL),
    COUNT_IF(_source_file_modification_ts IS NULL),
    COUNT_IF(_ingestion_ts IS NULL),
    COUNT_IF(_ingestion_date IS NULL)
FROM clinical_trial_intelligence.bronze.safety_adverse_events

ORDER BY table_name;



-- ============================================================
-- 6. EDC SUBJECT SCHEMA INSPECTION
-- ============================================================

DESCRIBE TABLE clinical_trial_intelligence.bronze.edc_subjects;



-- ============================================================
-- 7. EDC SUBJECT BUSINESS-KEY PROFILE
-- ============================================================
--
-- Repeated subject_id values are not automatically considered
-- Bronze errors because Bronze can contain multiple source
-- records / versions for the same business entity.
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT subject_id) AS distinct_subject_ids,
    COUNT_IF(subject_id IS NULL) AS null_subject_ids
FROM clinical_trial_intelligence.bronze.edc_subjects;



-- ============================================================
-- 8. EDC SUBJECT VERSION / REPETITION PROFILE
-- ============================================================

SELECT
    subject_id,
    COUNT(*) AS version_count,
    MIN(_ingestion_ts) AS first_seen,
    MAX(_ingestion_ts) AS latest_seen
FROM clinical_trial_intelligence.bronze.edc_subjects
GROUP BY subject_id
HAVING COUNT(*) > 1
ORDER BY version_count DESC, subject_id;